# LeetCode 85 — Maximal Rectangle (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:

```

Đây là một bài **Hard** rất nổi tiếng vì nó dạy một kỹ thuật cực kỳ đáng giá: **biến bài toán 2 chiều (ma trận) thành nhiều bài toán 1 chiều (histogram)**. Khi đã thấy được "cây cầu" đó, bài Hard này trở thành: *lặp lại một bài quen thuộc `m` lần*.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đúng đề: phần tử là **chuỗi** `'0'`/`'1'` (không phải số!) và cách mô tả một hình chữ nhật bằng 4 con số.
2. **Cách 1 — Brute force** thử mọi hình chữ nhật, rồi tăng tốc phần kiểm tra bằng **prefix sum 2D** (tổng tiền tố 2 chiều) → `O(m²n²)`.
3. **Cách 2 — DP `width`**: đếm số `1` liên tiếp kết thúc tại mỗi ô, cố định góc dưới-phải rồi "lan lên trên" → `O(m²n)`.
4. **Cách 3 — Histogram + Monotonic stack** (cách chính, `O(mn)`): mỗi hàng là đáy của một biểu đồ cột; giải **Largest Rectangle in Histogram (LeetCode 84)** bằng **ngăn xếp đơn điệu** — dạy lại từ đầu.
5. **Cách 4 — DP `height / left / right`**: cách "maximal rectangle DP" nổi tiếng, cũng `O(mn)`, không cần stack.
6. Trace từng bước trên ví dụ của đề, vì sao đúng, bảng so sánh, lỗi hay gặp, mẹo nhận dạng, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho một ma trận nhị phân `matrix` kích thước `rows x cols`, chỉ chứa các ký tự `'0'` và `'1'`. Hãy tìm **hình chữ nhật lớn nhất chỉ chứa toàn `'1'`** và trả về **diện tích** của nó.

**Example 1**

```text
Input: matrix = [["1","0","1","0","0"],
                 ["1","0","1","1","1"],
                 ["1","1","1","1","1"],
                 ["1","0","0","1","0"]]
Output: 6
Explanation: The maximal rectangle is shown in the above picture.
```

**Example 2**

```text
Input: matrix = [["0"]]
Output: 0
```

**Example 3**

```text
Input: matrix = [["1"]]
Output: 1
```

**Constraints**

- `rows == matrix.length`
- `cols == matrix[i].length`
- `1 <= rows, cols <= 200`
- `matrix[i][j]` is `'0'` or `'1'`.

Những điều cần để ý ngay:

- "Hình chữ nhật" ở đây là một **khối ô liền nhau, cạnh song song với các cạnh ma trận** (không xoay nghiêng). Hình vuông cũng là hình chữ nhật.
- Trả về **diện tích** = số ô, không phải tọa độ.
- Ma trận có thể tới `200 x 200 = 40 000` ô. Ta sẽ dùng con số này để đánh giá cách nào chạy kịp.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import
import random
import time

matrix1 = [["1", "0", "1", "0", "0"],
           ["1", "0", "1", "1", "1"],
           ["1", "1", "1", "1", "1"],
           ["1", "0", "0", "1", "0"]]
matrix2 = [["0"]]
matrix3 = [["1"]]

print("Example 1:", len(matrix1), "hàng x", len(matrix1[0]), "cột -> mong đợi 6")
print("Example 2:", matrix2, "-> mong đợi 0")
print("Example 3:", matrix3, "-> mong đợi 1")

Example 1: 4 hàng x 5 cột -> mong đợi 6
Example 2: [['0']] -> mong đợi 0
Example 3: [['1']] -> mong đợi 1


## 2. Vẽ ma trận ra cho dễ nhìn

Ta mô tả một hình chữ nhật bằng **4 con số**: hàng trên `r1`, cột trái `c1`, hàng dưới `r2`, cột phải `c2` (tính cả hai đầu, đánh số từ 0).

- Số hàng của nó: `r2 - r1 + 1`.
- Số cột của nó: `c2 - c1 + 1`.
- Diện tích: `(r2 - r1 + 1) * (c2 - c1 + 1)`.

Ví dụ hình chữ nhật đáp án của example 1 là `r1=1, c1=2, r2=2, c2=4`: cao `2 - 1 + 1 = 2` hàng, rộng `4 - 2 + 1 = 3` cột → diện tích `6`.

Hàm vẽ dưới đây in `1` thành `1`, `0` thành `.` (cho dễ nhìn), và nếu truyền `rect` thì các ô trong hình chữ nhật được tô bằng `#`. Bạn không cần hiểu hết hàm vẽ; chỉ cần biết cách gọi.

In [2]:
def show_matrix(matrix, rect=None):
    """In ma trận; rect = (r1, c1, r2, c2) sẽ được tô bằng '#'."""
    n = len(matrix[0])
    print("    " + " ".join(str(j) for j in range(n)))   # chỉ số cột
    for i, row in enumerate(matrix):
        line = []
        for j, ch in enumerate(row):
            inside = rect is not None and rect[0] <= i <= rect[2] and rect[1] <= j <= rect[3]
            if inside:
                line.append("#")
            else:
                line.append("1" if ch == "1" else ".")
        print(f"{i} | " + " ".join(line))


print("Example 1:")
show_matrix(matrix1)
print()
print("Hình chữ nhật đáp án (r1=1, c1=2, r2=2, c2=4), diện tích 6:")
show_matrix(matrix1, rect=(1, 2, 2, 4))

Example 1:
    0 1 2 3 4
0 | 1 . 1 . .
1 | 1 . 1 1 1
2 | 1 1 1 1 1
3 | 1 . . 1 .

Hình chữ nhật đáp án (r1=1, c1=2, r2=2, c2=4), diện tích 6:
    0 1 2 3 4
0 | 1 . 1 . .
1 | 1 . # # #
2 | 1 1 # # #
3 | 1 . . 1 .


### Bẫy số 1 của bài: phần tử là **chuỗi**, không phải số

Trong Python, **mọi chuỗi khác rỗng đều "truthy"** (được coi là `True` khi đặt trong `if`), kể cả chuỗi `"0"`. Vì thế viết `if matrix[i][j]:` sẽ coi **mọi ô** là `1` — sai hoàn toàn.

Hai cách đúng:

- So sánh trực tiếp: `matrix[i][j] == "1"`.
- Đổi ra số: `int(matrix[i][j])` cho `0` hoặc `1`.

In [3]:
cell = "0"
print("bool('0')      =", bool(cell))        # True  <- bẫy!
print("cell == '1'    =", cell == "1")       # False <- đúng
print("int('0')       =", int(cell))         # 0
print("int('1') + 1   =", int("1") + 1)      # 2
if cell:
    print("if '0': ... vẫn chạy vào đây, vì chuỗi '0' khác rỗng!")

bool('0')      = True
cell == '1'    = False
int('0')       = 0
int('1') + 1   = 2
if '0': ... vẫn chạy vào đây, vì chuỗi '0' khác rỗng!


## 3. Cách 1 — Brute force: thử mọi hình chữ nhật

Cách mà ai mới học cũng nghĩ ra đầu tiên:

1. Chọn góc trên-trái `(r1, c1)`.
2. Chọn góc dưới-phải `(r2, c2)` với `r2 >= r1`, `c2 >= c1`.
3. Kiểm tra **từng ô** bên trong: nếu tất cả là `'1'` thì cập nhật đáp án bằng diện tích.

Bao nhiêu hình chữ nhật? Chọn cặp hàng `(r1, r2)` có `m(m+1)/2` cách, chọn cặp cột có `n(n+1)/2` cách. Với `m = n = 200`:

`(200·201/2)² = 20100² ≈ 4 × 10⁸` hình chữ nhật — mà mỗi cái còn phải kiểm tra tới hàng chục nghìn ô. Quá chậm, nhưng là **"chuẩn vàng"** để đối chiếu các cách nhanh sau này.

In [4]:
def count_rectangles(m, n):
    return (m * (m + 1) // 2) * (n * (n + 1) // 2)

print("Số hình chữ nhật trong ma trận 4x5    :", count_rectangles(4, 5))
print("Số hình chữ nhật trong ma trận 200x200:", count_rectangles(200, 200))


def maximal_rectangle_bruteforce(matrix: List[List[str]]) -> int:
    m, n = len(matrix), len(matrix[0])
    best = 0
    for r1 in range(m):
        for c1 in range(n):
            for r2 in range(r1, m):
                for c2 in range(c1, n):
                    # kiểm tra mọi ô trong hình chữ nhật (r1, c1) - (r2, c2)
                    all_ones = all(matrix[i][j] == "1"
                                   for i in range(r1, r2 + 1)
                                   for j in range(c1, c2 + 1))
                    if all_ones:
                        best = max(best, (r2 - r1 + 1) * (c2 - c1 + 1))
    return best

print(maximal_rectangle_bruteforce(matrix1))   # 6
print(maximal_rectangle_bruteforce(matrix2))   # 0
print(maximal_rectangle_bruteforce(matrix3))   # 1

Số hình chữ nhật trong ma trận 4x5    : 150
Số hình chữ nhật trong ma trận 200x200: 404010000
6
0
1


### Cách 1 vướng ở đâu?

Có **hai** vòng lặp tốn kém chồng lên nhau:

- Liệt kê hình chữ nhật: `O(m²n²)`.
- Kiểm tra mỗi hình: tới `O(mn)` ô.

Tổng cộng cỡ `O(m³n³)`. Phần **kiểm tra** có thể làm nhanh hơn nhiều: thay vì đếm từng ô, ta muốn hỏi "**tổng các ô trong hình chữ nhật là bao nhiêu?**" trong `O(1)`. Nếu tổng bằng đúng diện tích → toàn `1`. Công cụ cho việc này là **prefix sum** (tổng tiền tố).

### Kiến thức nền: prefix sum 1 chiều

Cho mảng `a = [3, 1, 4, 1, 5]`. Ta dựng `P` dài hơn `a` một phần tử, với `P[0] = 0` và `P[k] = a[0] + ... + a[k-1]`:

| k | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| P[k] | 0 | 3 | 4 | 8 | 9 | 14 |

Khi đó tổng đoạn `a[l..r]` (tính cả hai đầu) = `P[r+1] - P[l]`. Ví dụ `a[1..3] = 1 + 4 + 1 = 6 = P[4] - P[1] = 9 - 3`. Chỉ **một phép trừ**, dù đoạn dài bao nhiêu.

In [5]:
a = [3, 1, 4, 1, 5]
P = [0] * (len(a) + 1)
for k in range(len(a)):
    P[k + 1] = P[k] + a[k]
print("P =", P)

l, r = 1, 3
print(f"sum(a[{l}..{r}]) bằng cộng tay  :", sum(a[l:r + 1]))
print(f"sum(a[{l}..{r}]) bằng prefix sum:", P[r + 1] - P[l])

P = [0, 3, 4, 8, 9, 14]
sum(a[1..3]) bằng cộng tay  : 6
sum(a[1..3]) bằng prefix sum: 6


### Prefix sum 2 chiều

Mở rộng lên ma trận: `P[i][j]` = tổng mọi ô trong hình chữ nhật từ góc `(0, 0)` tới `(i-1, j-1)`. Bảng `P` có kích thước `(m+1) x (n+1)`, hàng 0 và cột 0 toàn số 0 (cho khỏi phải xử lý biên).

**Cách dựng** (nguyên lý bao hàm – loại trừ):

```text
P[i+1][j+1] = cell(i, j) + P[i][j+1] + P[i+1][j] - P[i][j]
```

Lý do: `P[i][j+1]` (khối phía trên) và `P[i+1][j]` (khối bên trái) **cùng chứa** khối `P[i][j]` ở góc trên-trái, nên bị cộng 2 lần → trừ đi 1 lần.

**Cách hỏi** tổng hình chữ nhật `(r1, c1) - (r2, c2)`:

```text
sum = P[r2+1][c2+1] - P[r1][c2+1] - P[r2+1][c1] + P[r1][c1]
```

Lấy khối lớn, bỏ dải phía trên, bỏ dải bên trái, rồi cộng lại góc trên-trái đã bị trừ 2 lần.

In [6]:
def build_prefix_2d(matrix):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]      # (m+1) x (n+1), toàn 0
    for i in range(m):
        for j in range(n):
            cell = 1 if matrix[i][j] == "1" else 0
            P[i + 1][j + 1] = cell + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P


def rect_sum(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]


P1 = build_prefix_2d(matrix1)
print("Bảng P của example 1 (hàng 0 và cột 0 là biên toàn 0):")
for row in P1:
    print("  ", row)

print("Tổng hình (1,2)-(2,4):", rect_sum(P1, 1, 2, 2, 4), "| diện tích:", 2 * 3, "-> toàn 1")
print("Tổng hình (0,0)-(2,2):", rect_sum(P1, 0, 0, 2, 2), "| diện tích:", 3 * 3, "-> có ô 0")

Bảng P của example 1 (hàng 0 và cột 0 là biên toàn 0):
   [0, 0, 0, 0, 0, 0]
   [0, 1, 1, 2, 2, 2]
   [0, 2, 2, 4, 5, 6]
   [0, 3, 4, 7, 9, 11]
   [0, 4, 5, 8, 11, 13]
Tổng hình (1,2)-(2,4): 6 | diện tích: 6 -> toàn 1
Tổng hình (0,0)-(2,2): 7 | diện tích: 9 -> có ô 0


### Brute force + prefix sum 2D

Giữ nguyên 4 vòng lặp liệt kê hình chữ nhật, nhưng thay bước kiểm tra `O(mn)` bằng `rect_sum` `O(1)`:

- **Time** `O(m²n²)`: với `200 x 200` là cỡ `4 × 10⁸` lần hỏi — vẫn **TLE** (Time Limit Exceeded, quá thời gian) trong Python.
- **Space** `O(mn)` cho bảng `P`.

Một mẹo nhỏ: nếu hình `(r1,c1)-(r2,c2)` đã chứa ô `0` thì mở rộng `c2` thêm nữa cũng vẫn chứa ô `0` → có thể `break`. Mẹo này giúp chạy nhanh hơn trên thực tế nhưng không đổi worst case (ma trận toàn `1`).

In [7]:
def maximal_rectangle_prefix2d(matrix: List[List[str]]) -> int:
    m, n = len(matrix), len(matrix[0])
    P = build_prefix_2d(matrix)
    best = 0
    for r1 in range(m):
        for c1 in range(n):
            for r2 in range(r1, m):
                for c2 in range(c1, n):
                    area = (r2 - r1 + 1) * (c2 - c1 + 1)
                    if rect_sum(P, r1, c1, r2, c2) != area:
                        break                     # mở rộng c2 thêm cũng vẫn chứa ô 0
                    best = max(best, area)
    return best

print(maximal_rectangle_prefix2d(matrix1))   # 6
print(maximal_rectangle_prefix2d(matrix2))   # 0
print(maximal_rectangle_prefix2d(matrix3))   # 1

6
0
1


## 4. Cách 2 — DP `width`: cố định góc dưới-phải rồi lan lên trên

### Nút thắt của cách 1

Cách 1 đối xử với mọi hình chữ nhật như nhau, kể cả hàng trăm triệu hình chắc chắn chứa số `0`. Ta muốn **tái sử dụng thông tin** giữa các ô cạnh nhau — đó là tinh thần **DP** (Dynamic Programming, quy hoạch động: lưu kết quả bài toán con để dùng lại).

### Định nghĩa `width[i][j]`

`width[i][j]` = **số ô `1` liên tiếp trên hàng `i`, kết thúc đúng tại cột `j`** (đếm từ `j` sang trái cho tới khi gặp `0`).

Công thức chỉ cần nhìn ô bên trái:

```text
width[i][j] = 0                       nếu matrix[i][j] == '0'
width[i][j] = width[i][j-1] + 1       nếu matrix[i][j] == '1'   (với j = 0 thì là 1)
```

Ví dụ hàng `"10111"`: đọc từ trái sang → `1, 0, 1, 2, 3`.

In [8]:
row = ["1", "0", "1", "1", "1"]
w = []
run = 0
for ch in row:
    run = run + 1 if ch == "1" else 0     # gặp '0' thì đếm lại từ 0
    w.append(run)
print("hàng :", row)
print("width:", w)


def build_width(matrix):
    m, n = len(matrix), len(matrix[0])
    width = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                width[i][j] = (width[i][j - 1] if j > 0 else 0) + 1
    return width


print("\nBảng width của example 1:")
for i, r in enumerate(build_width(matrix1)):
    print(f"  hàng {i}: {r}")

hàng : ['1', '0', '1', '1', '1']
width: [1, 0, 1, 2, 3]

Bảng width của example 1:
  hàng 0: [1, 0, 1, 0, 0]
  hàng 1: [1, 0, 1, 2, 3]
  hàng 2: [1, 2, 3, 4, 5]
  hàng 3: [1, 0, 0, 1, 0]


### Ý tưởng: lan lên trên và lấy `min`

Cố định góc **dưới-phải** tại `(i, j)`. Một hình chữ nhật có góc dưới-phải này và cao `h` hàng (từ hàng `i-h+1` tới `i`) chỉ rộng được tối đa bằng **hàng hẹp nhất** trong số đó:

```text
rộng tối đa = min(width[k][j]  với k = i-h+1 .. i)
diện tích   = rộng tối đa * h
```

Vậy ta đi `k = i, i-1, i-2, ...` lên trên, giữ `min_w` = min các `width[k][j]` đã gặp, mỗi bước tính `min_w * (i - k + 1)`. Khi `min_w` về `0` thì dừng (lên cao nữa cũng chỉ rộng 0).

Ví dụ ô `(2, 4)` trong example 1, cột 4 của bảng `width` là `[0, 3, 5, 0]` (hàng 0..3):

| k | width[k][4] | min_w | chiều cao `i-k+1` | diện tích | best |
|---:|---:|---:|---:|---:|---:|
| 2 | 5 | 5 | 1 | 5 | 5 |
| 1 | 3 | 3 | 2 | **6** | 6 |
| 0 | 0 | 0 | 3 | 0 → dừng | 6 |

Đây chính là hình chữ nhật đáp án `(1,2)-(2,4)`: cao 2, rộng 3.

In [9]:
def maximal_rectangle_width_dp(matrix: List[List[str]]) -> int:
    m, n = len(matrix), len(matrix[0])
    width = build_width(matrix)
    best = 0
    for i in range(m):
        for j in range(n):
            min_w = width[i][j]
            k = i
            while k >= 0 and min_w > 0:          # lan lên trên
                min_w = min(min_w, width[k][j])
                best = max(best, min_w * (i - k + 1))
                k -= 1
    return best

print(maximal_rectangle_width_dp(matrix1))   # 6
print(maximal_rectangle_width_dp(matrix2))   # 0
print(maximal_rectangle_width_dp(matrix3))   # 1

6
0
1


### Trace cách 2 tại góc dưới-phải `(2, 4)`

Hàm `trace_width_dp` in đúng các bước ở bảng trên. Để ý nhánh đặc biệt: khi `width[k][j] = 0` thì `min_w` thành 0 và vòng `while` dừng.

In [10]:
def trace_width_dp(matrix, i, j):
    width = build_width(matrix)
    print(f"Góc dưới-phải ({i}, {j}); cột {j} của width = {[width[k][j] for k in range(len(matrix))]}")
    min_w = width[i][j]
    best = 0
    k = i
    while k >= 0 and min_w > 0:
        min_w = min(min_w, width[k][j])
        h = i - k + 1
        area = min_w * h
        best = max(best, area)
        print(f"  k={k}: width[{k}][{j}]={width[k][j]}, min_w={min_w}, cao={h}, diện tích={area}, best={best}")
        k -= 1
    print("  dừng vì", "min_w = 0" if min_w == 0 else "đã lên tới hàng 0")
    return best

trace_width_dp(matrix1, 2, 4)

Góc dưới-phải (2, 4); cột 4 của width = [0, 3, 5, 0]
  k=2: width[2][4]=5, min_w=5, cao=1, diện tích=5, best=5
  k=1: width[1][4]=3, min_w=3, cao=2, diện tích=6, best=6
  k=0: width[0][4]=0, min_w=0, cao=3, diện tích=0, best=6
  dừng vì min_w = 0


6

### Độ phức tạp cách 2 và nút thắt còn lại

Gọi `m` = số hàng, `n` = số cột.

- Dựng `width`: `O(mn)`.
- Mỗi ô `(i, j)` lan lên tối đa `i + 1` hàng → tổng `O(m²n)`. Với `200 x 200` là cỡ `8 × 10⁶` bước: **chạy kịp**, đã tốt hơn cách 1 rất nhiều.
- Space `O(mn)` cho bảng `width` (có thể giảm, nhưng không phải trọng tâm).

Nút thắt: với mỗi ô ta vẫn **lan lên lại từ đầu**, lặp lại rất nhiều phép `min` giống nhau giữa các ô cùng cột. Câu hỏi: có góc nhìn nào để mỗi ô chỉ được "xử lý" **hằng số lần** không? → Đổi góc nhìn: thay vì đếm theo **chiều ngang** rồi lan **lên**, ta đếm theo **chiều dọc** rồi giải **ngang** một lần cho cả hàng.

## 5. Insight chính: mỗi hàng là đáy của một **histogram**

**Histogram** (biểu đồ cột) là một dãy cột đứng sát nhau, mỗi cột rộng 1, cao `heights[j]`.

Chọn hàng `i` làm **đáy**. Với mỗi cột `j`, đặt:

```text
heights[j] = số ô '1' liên tiếp tính từ hàng i đi NGƯỢC LÊN trên (dừng khi gặp '0')
```

Cập nhật từ hàng trên xuống cực kỳ đơn giản:

```text
heights[j] = heights[j] + 1   nếu matrix[i][j] == '1'
heights[j] = 0                nếu matrix[i][j] == '0'   (cột bị "cắt đứt")
```

Với example 1:

| đáy là hàng | hàng gốc | heights |
|---|---|---|
| 0 | `1 0 1 0 0` | `[1, 0, 1, 0, 0]` |
| 1 | `1 0 1 1 1` | `[2, 0, 2, 1, 1]` |
| 2 | `1 1 1 1 1` | `[3, 1, 3, 2, 2]` |
| 3 | `1 0 0 1 0` | `[4, 0, 0, 3, 0]` |

**Mấu chốt:** mọi hình chữ nhật toàn `1` đều có **một hàng đáy** `i`. Nếu nó chiếm các cột `c1..c2` và cao `h`, thì mọi cột trong đó đều có `heights[j] >= h` → nó là một hình chữ nhật nằm gọn **bên trong histogram của hàng `i`**. Ngược lại, mọi hình chữ nhật trong histogram đều là khối toàn `1` trong ma trận.

Vậy:

```text
đáp án = max over i của  (hình chữ nhật lớn nhất trong histogram của hàng i)
```

Bài con "hình chữ nhật lớn nhất trong histogram" chính là **LeetCode 84 — Largest Rectangle in Histogram**. Ta sẽ học nó từ đầu ngay sau đây.

In [11]:
def draw_histogram(heights):
    top = max(heights) if heights else 0
    for level in range(top, 0, -1):
        print(f"{level} | " + " ".join("#" if h >= level else " " for h in heights))
    print("  +" + "--" * len(heights))
    print("    " + " ".join(str(j) for j in range(len(heights))))


heights = [0] * len(matrix1[0])
for i, row in enumerate(matrix1):
    for j, ch in enumerate(row):
        heights[j] = heights[j] + 1 if ch == "1" else 0
    print(f"Đáy hàng {i}: heights = {heights}")

print("\nHistogram khi đáy là hàng 2, heights = [3, 1, 3, 2, 2]:")
draw_histogram([3, 1, 3, 2, 2])

Đáy hàng 0: heights = [1, 0, 1, 0, 0]
Đáy hàng 1: heights = [2, 0, 2, 1, 1]
Đáy hàng 2: heights = [3, 1, 3, 2, 2]
Đáy hàng 3: heights = [4, 0, 0, 3, 0]

Histogram khi đáy là hàng 2, heights = [3, 1, 3, 2, 2]:
3 | #   #    
2 | #   # # #
1 | # # # # #
  +----------
    0 1 2 3 4


### Bài con: hình chữ nhật lớn nhất trong histogram — cách dễ nghĩ

Nhìn histogram `[3, 1, 3, 2, 2]` ở trên. Một hình chữ nhật trong histogram bị giới hạn chiều cao bởi **cột thấp nhất** mà nó đi qua.

Đảo ngược câu hỏi: **với mỗi cột `j`, giả sử nó là cột thấp nhất của hình chữ nhật**, thì hình đó cao `heights[j]` và **mở rộng sang trái, sang phải được bao xa?** — tới khi gặp một cột **thấp hơn** `heights[j]` thì phải dừng.

```text
left  = index cột gần nhất bên TRÁI có chiều cao < heights[j]   (không có thì -1)
right = index cột gần nhất bên PHẢI có chiều cao < heights[j]   (không có thì n)
width = right - left - 1
area  = heights[j] * width
```

Ví dụ `j = 3` (`heights[3] = 2`): bên trái gần nhất thấp hơn 2 là cột 1 (cao 1) → `left = 1`; bên phải không có cột nào thấp hơn → `right = 5`. `width = 5 - 1 - 1 = 3`, `area = 2 * 3 = 6`. Chính là hình chữ nhật đáp án!

Cách dễ nghĩ: với mỗi `j`, dùng hai vòng `while` bò sang hai bên → `O(n²)` cho một histogram.

In [12]:
def largest_rectangle_histogram_bruteforce(heights: List[int]) -> int:
    n = len(heights)
    best = 0
    for j in range(n):
        left = j - 1
        while left >= 0 and heights[left] >= heights[j]:
            left -= 1
        right = j + 1
        while right < n and heights[right] >= heights[j]:
            right += 1
        best = max(best, heights[j] * (right - left - 1))
    return best

print(largest_rectangle_histogram_bruteforce([3, 1, 3, 2, 2]))      # 6
print(largest_rectangle_histogram_bruteforce([2, 1, 5, 6, 2, 3]))   # 10 (example của LeetCode 84)

6
10


Hai vòng `while` bò sang hai bên là phần lặp lại tốn kém. Câu hỏi "**cột gần nhất bên trái/phải thấp hơn tôi**" có một công cụ chuyên trị trong `O(n)` cho cả mảng: **monotonic stack**. Trước tiên ta ôn lại stack.

### Kiến thức nền: stack (ngăn xếp)

**Stack** giống một chồng đĩa: đĩa đặt vào **sau cùng** được lấy ra **đầu tiên** — gọi là **LIFO** (Last In, First Out). Trong Python dùng luôn `list`:

| Thao tác | Code | Ý nghĩa |
|---|---|---|
| push (đặt lên đỉnh) | `stack.append(x)` | thêm vào cuối list, `O(1)` |
| pop (lấy ra từ đỉnh) | `stack.pop()` | bỏ và trả về phần tử cuối, `O(1)` |
| peek (nhìn đỉnh) | `stack[-1]` | xem phần tử cuối, không bỏ |
| rỗng chưa? | `if not stack:` | list rỗng thì `not stack` là `True` |

In [13]:
stack = []
stack.append(10)
stack.append(20)
stack.append(30)
print("stack sau 3 lần push:", stack)
print("đỉnh stack[-1]     :", stack[-1])
print("pop ra             :", stack.pop())
print("stack còn lại      :", stack)
print("stack rỗng?        :", not stack)

stack sau 3 lần push: [10, 20, 30]
đỉnh stack[-1]     : 30
pop ra             : 30
stack còn lại      : [10, 20]
stack rỗng?        : False


### Kiến thức nền: monotonic stack (ngăn xếp đơn điệu)

**Monotonic increasing stack** là stack mà các giá trị (ở đây: chiều cao của các index được lưu) **tăng dần từ đáy lên đỉnh**. Quy tắc giữ tính "tăng dần":

> Trước khi push index `i`, **pop hết** các index ở đỉnh có chiều cao `>= heights[i]`.

Sau khi pop xong, phần tử còn ở đỉnh (nếu có) chính là **cột gần nhất bên trái thấp hơn hẳn** `heights[i]`. Vì sao? Những cột bị pop đều cao hơn hoặc bằng `heights[i]` — chúng không thể là "cột thấp hơn" của `i`, **và** cũng không bao giờ là đáp án cho các cột đến sau `i` nữa (vì `i` vừa gần hơn vừa thấp hơn hoặc bằng chúng). Nên vứt đi là an toàn.

Ví dụ mini với `a = [3, 1, 3, 2, 2]`, tìm "cột thấp hơn gần nhất bên trái" (`prev_smaller`):

In [14]:
a = [3, 1, 3, 2, 2]
prev_smaller = []
stack = []                      # lưu index, chiều cao tăng dần từ đáy lên đỉnh
for i, x in enumerate(a):       # enumerate cho cả index i lẫn giá trị x
    while stack and a[stack[-1]] >= x:
        stack.pop()
    prev_smaller.append(stack[-1] if stack else -1)
    stack.append(i)
    print(f"i={i} (h={x}): prev_smaller={prev_smaller[-1]:>2}, stack sau push = {stack}")
print("prev_smaller =", prev_smaller)

i=0 (h=3): prev_smaller=-1, stack sau push = [0]
i=1 (h=1): prev_smaller=-1, stack sau push = [1]
i=2 (h=3): prev_smaller= 1, stack sau push = [1, 2]
i=3 (h=2): prev_smaller= 1, stack sau push = [1, 3]
i=4 (h=2): prev_smaller= 1, stack sau push = [1, 4]
prev_smaller = [-1, -1, 1, 1, 1]


Làm y hệt nhưng duyệt **từ phải sang trái** sẽ ra `next_smaller` (cột thấp hơn gần nhất bên phải, không có thì `n`). Có đủ hai mảng là tính được diện tích của **mọi** cột trong `O(n)`:

```text
area[j] = heights[j] * (next_smaller[j] - prev_smaller[j] - 1)
```

Mỗi index được push đúng 1 lần và pop nhiều nhất 1 lần → mỗi lượt duyệt là `O(n)`, dù có vòng `while` lồng bên trong.

In [15]:
def largest_rectangle_two_pass(heights: List[int]) -> int:
    n = len(heights)
    prev_smaller, next_smaller = [-1] * n, [n] * n
    stack = []
    for i in range(n):                         # trái -> phải
        while stack and heights[stack[-1]] >= heights[i]:
            stack.pop()
        prev_smaller[i] = stack[-1] if stack else -1
        stack.append(i)
    stack = []
    for i in range(n - 1, -1, -1):             # phải -> trái
        while stack and heights[stack[-1]] >= heights[i]:
            stack.pop()
        next_smaller[i] = stack[-1] if stack else n
        stack.append(i)
    areas = [heights[j] * (next_smaller[j] - prev_smaller[j] - 1) for j in range(n)]
    print("prev_smaller:", prev_smaller)
    print("next_smaller:", next_smaller)
    print("areas       :", areas)
    return max(areas) if areas else 0

print("best =", largest_rectangle_two_pass([3, 1, 3, 2, 2]))

prev_smaller: [-1, -1, 1, 1, 1]
next_smaller: [1, 5, 3, 5, 5]
areas       : [3, 5, 3, 6, 6]
best = 6


### Gộp thành một lượt duyệt: "khi bị pop, tôi biết cả hai biên"

Nhìn lại quy tắc pop: index `top` bị pop **bởi** `i` khi `heights[i] < heights[top]` (ở phiên bản một lượt ta dùng `<` chặt, lý do ở dưới). Lúc đó:

- **Biên phải** của `top` = `i` (cột đầu tiên bên phải thấp hơn nó — vì nếu có cột thấp hơn sớm hơn, `top` đã bị pop sớm hơn).
- **Biên trái** của `top` = phần tử nằm **ngay dưới** nó trong stack (sau khi pop, chính là `stack[-1]`), hoặc `-1` nếu stack rỗng.

Vậy ngay lúc pop, ta tính luôn:

```text
h     = heights[top]
width = i - stack[-1] - 1     (nếu stack còn phần tử)
width = i                     (nếu stack rỗng: kéo dài tới tận cột 0)
```

**Sentinel (lính canh):** những cột còn kẹt trong stack ở cuối mảng thì chưa bao giờ bị pop. Mẹo: nối thêm một cột cao `0` vào cuối — nó thấp hơn mọi cột nên sẽ "dọn sạch" stack.

**Cột bằng nhau:** với `heights = [.., 2, 2]`, cột 2 thứ hai **không** pop cột 2 thứ nhất (vì dùng `<` chặt, mà `2 < 2` là sai). Cột sau sẽ bị pop trước với width nhỏ; nhưng cột trước bị pop sau với **width đầy đủ** — nên đáp án vẫn đúng. (Dùng `<=` cũng ra đúng đáp án, chỉ khác ở việc cột nào "nhận" diện tích lớn nhất.)

In [16]:
def largest_rectangle_histogram(heights: List[int]) -> int:
    stack = []                                  # index, chiều cao tăng dần (không giảm) từ đáy lên đỉnh
    best = 0
    extended = heights + [0]                    # sentinel cao 0 ở cuối
    for i, h in enumerate(extended):
        while stack and extended[stack[-1]] > h:
            top = stack.pop()
            width = i - stack[-1] - 1 if stack else i
            best = max(best, extended[top] * width)
        stack.append(i)
    return best

print(largest_rectangle_histogram([3, 1, 3, 2, 2]))      # 6
print(largest_rectangle_histogram([2, 1, 5, 6, 2, 3]))   # 10
print(largest_rectangle_histogram([2, 4]))               # 4
print(largest_rectangle_histogram([0, 0]))               # 0

6
10
4
0


### Trace monotonic stack trên `heights = [3, 1, 3, 2, 2]` (đáy hàng 2 của example 1)

Có sentinel nên mảng thực sự duyệt là `[3, 1, 3, 2, 2, 0]`.

| i | h | pop top (h) | biên trái | width | diện tích | best | stack sau bước |
|---:|---:|---|---|---:|---:|---:|---|
| 0 | 3 | — | | | | 0 | `[0]` |
| 1 | 1 | 0 (3) | rỗng → -1 | 1 | 3 | 3 | `[1]` |
| 2 | 3 | — | | | | 3 | `[1, 2]` |
| 3 | 2 | 2 (3) | 1 | 1 | 3 | 3 | `[1, 3]` |
| 4 | 2 | — (2 > 2 sai) | | | | 3 | `[1, 3, 4]` |
| 5 | 0 | 4 (2) | 3 | 1 | 2 | 3 | |
| 5 | 0 | 3 (2) | 1 | 3 | **6** | 6 | |
| 5 | 0 | 1 (1) | rỗng → -1 | 5 | 5 | 6 | `[5]` |

Hai nhánh đáng chú ý:

- `i = 4`: hai cột bằng nhau, **không** pop. Đến `i = 5` cột 4 bị pop với width 1 (diện tích 2), nhưng ngay sau đó cột 3 bị pop với width 3 → diện tích 6.
- Stack rỗng sau khi pop (ở `i = 1` và lần pop cuối ở `i = 5`): width bằng chính `i`, nghĩa là hình chữ nhật kéo dài tới tận cột 0.

In [17]:
def trace_histogram_stack(heights):
    extended = heights + [0]
    stack, best = [], 0
    print(f"heights + sentinel = {extended}")
    for i, h in enumerate(extended):
        while stack and extended[stack[-1]] > h:
            top = stack.pop()
            left = stack[-1] if stack else -1
            width = i - left - 1
            area = extended[top] * width
            best = max(best, area)
            print(f"  i={i} h={h}: pop {top} (h={extended[top]}), biên trái={left:>2}, "
                  f"width={width}, diện tích={area}, best={best}")
        stack.append(i)
        print(f"  i={i} h={h}: push -> stack = {stack}")
    return best

print("Kết quả:", trace_histogram_stack([3, 1, 3, 2, 2]))

heights + sentinel = [3, 1, 3, 2, 2, 0]
  i=0 h=3: push -> stack = [0]
  i=1 h=1: pop 0 (h=3), biên trái=-1, width=1, diện tích=3, best=3
  i=1 h=1: push -> stack = [1]
  i=2 h=3: push -> stack = [1, 2]
  i=3 h=2: pop 2 (h=3), biên trái= 1, width=1, diện tích=3, best=3
  i=3 h=2: push -> stack = [1, 3]
  i=4 h=2: push -> stack = [1, 3, 4]
  i=5 h=0: pop 4 (h=2), biên trái= 3, width=1, diện tích=2, best=3
  i=5 h=0: pop 3 (h=2), biên trái= 1, width=3, diện tích=6, best=6
  i=5 h=0: pop 1 (h=1), biên trái=-1, width=5, diện tích=5, best=6
  i=5 h=0: push -> stack = [5]
Kết quả: 6


### Ghép lại: Cách 3 — Histogram + Monotonic stack cho cả ma trận

```text
heights = [0] * n
với mỗi hàng i:
    cập nhật heights theo hàng i     (+1 nếu '1', về 0 nếu '0')
    best = max(best, largest_rectangle_histogram(heights))
```

Code ngắn gọn bất ngờ, vì phần "khó" đã gói trong hàm histogram.

In [18]:
def maximal_rectangle_histogram(matrix: List[List[str]]) -> int:
    n = len(matrix[0])
    heights = [0] * n
    best = 0
    for row in matrix:
        for j in range(n):
            heights[j] = heights[j] + 1 if row[j] == "1" else 0
        best = max(best, largest_rectangle_histogram(heights))
    return best

print(maximal_rectangle_histogram(matrix1))   # 6
print(maximal_rectangle_histogram(matrix2))   # 0
print(maximal_rectangle_histogram(matrix3))   # 1

6
0
1


### Trace cách 3 trên example 1 (theo từng hàng đáy)

| đáy hàng | heights | hình lớn nhất trong histogram | best toàn cục |
|---:|---|---:|---:|
| 0 | `[1, 0, 1, 0, 0]` | 1 | 1 |
| 1 | `[2, 0, 2, 1, 1]` | 3 (cao 1, rộng 3: cột 2..4) | 3 |
| 2 | `[3, 1, 3, 2, 2]` | **6** (cao 2, rộng 3: cột 2..4) | 6 |
| 3 | `[4, 0, 0, 3, 0]` | 4 (cao 4, rộng 1: cột 0) | 6 |

Để ý hàng 3: cột 0 cao 4 cho diện tích 4 — một ứng viên mạnh, nhưng vẫn thua 6.

In [19]:
def trace_maximal_rectangle_histogram(matrix):
    n = len(matrix[0])
    heights = [0] * n
    best = 0
    for i, row in enumerate(matrix):
        for j in range(n):
            heights[j] = heights[j] + 1 if row[j] == "1" else 0
        row_best = largest_rectangle_histogram(heights)
        best = max(best, row_best)
        print(f"đáy hàng {i}: {''.join(row)} -> heights={heights}, lớn nhất trong histogram={row_best}, best={best}")
    return best

print("Kết quả:", trace_maximal_rectangle_histogram(matrix1))

đáy hàng 0: 10100 -> heights=[1, 0, 1, 0, 0], lớn nhất trong histogram=1, best=1
đáy hàng 1: 10111 -> heights=[2, 0, 2, 1, 1], lớn nhất trong histogram=3, best=3
đáy hàng 2: 11111 -> heights=[3, 1, 3, 2, 2], lớn nhất trong histogram=6, best=6
đáy hàng 3: 10010 -> heights=[4, 0, 0, 3, 0], lớn nhất trong histogram=4, best=6
Kết quả: 6


### Độ phức tạp cách 3

`m` = số hàng, `n` = số cột.

- Mỗi hàng: cập nhật `heights` `O(n)` + giải histogram `O(n)` (mỗi index push 1 lần, pop tối đa 1 lần — đây là lập luận **amortized** (khấu hao): vòng `while` bên trong tổng cộng chạy không quá `n + 1` lần cho cả hàng).
- Tổng **Time `O(mn)`** — với `200 x 200` chỉ cỡ `4 × 10⁴ × hằng số` bước.
- **Space `O(n)`**: mảng `heights` và stack. Ta không cần lưu cả ma trận phụ.

Đây là lời giải chính nên nộp.

## 6. Cách 4 — DP `height / left / right` (không cần stack)

Cách này cũng `O(mn)` nhưng tư duy khác: thay vì dùng stack để tìm biên trái/phải của mỗi cột trong histogram, ta **cập nhật biên trái/phải từ hàng trên xuống** bằng DP.

Với hàng đáy hiện tại `i` và mỗi cột `j`, duy trì 3 mảng:

| mảng | ý nghĩa |
|---|---|
| `height[j]` | như `heights` ở cách 3: số `1` liên tiếp kéo lên từ `(i, j)` |
| `left[j]` | cột trái nhất mà "thanh đứng" cao `height[j]` tại cột `j` có thể lan sang (tính cả) |
| `right[j]` | cột **ngay sau** cột phải nhất mà thanh đó lan tới (biên phải **không** tính) |

Diện tích ứng viên tại `(i, j)`: `(right[j] - left[j]) * height[j]`.

### Cập nhật cho hàng `i`

Đi từ trái sang phải với `cur_left` = cột bắt đầu của **đoạn `1` liên tiếp hiện tại** trên hàng `i`:

```text
nếu matrix[i][j] == '1':  left[j] = max(left[j], cur_left)   # bị chặn bởi hàng trên HOẶC hàng hiện tại, lấy cái chặt hơn
nếu matrix[i][j] == '0':  left[j] = 0;  cur_left = j + 1      # reset; đoạn 1 kế tiếp bắt đầu sau ô 0
```

Đi từ phải sang trái với `cur_right` = biên phải (không tính) của đoạn `1` hiện tại:

```text
nếu matrix[i][j] == '1':  right[j] = min(right[j], cur_right)
nếu matrix[i][j] == '0':  right[j] = n;  cur_right = j
```

**Tại sao reset `left[j] = 0` và `right[j] = n` ở ô `0`?** Vì ở ô `0` thì `height[j] = 0`, thanh đứng bị cắt đứt. Hàng sau, nếu cột `j` lại là `1`, thanh mới bắt đầu từ đầu và **không** nên bị giới hạn bởi hàng cũ. Đặt `0` và `n` là các giá trị "trung tính" nhất cho `max` và `min` (không chặn gì cả).

In [20]:
# Ví dụ mini: cur_left trên một hàng (chỉ phần "đoạn 1 hiện tại", chưa max với hàng trên)
row = ["1", "0", "1", "1", "1"]
cur_left = 0
run_left = []
for j, ch in enumerate(row):
    if ch == "1":
        run_left.append(cur_left)        # đoạn 1 chứa j bắt đầu ở cur_left
    else:
        run_left.append("-")             # ô 0: không thuộc đoạn nào
        cur_left = j + 1
print("hàng           :", row)
print("đoạn 1 bắt đầu :", run_left)

hàng           : ['1', '0', '1', '1', '1']
đoạn 1 bắt đầu : [0, '-', 2, 2, 2]


In [21]:
def maximal_rectangle_dp_hlr(matrix: List[List[str]]) -> int:
    m, n = len(matrix), len(matrix[0])
    height = [0] * n
    left = [0] * n          # 0 = không chặn gì bên trái
    right = [n] * n         # n = không chặn gì bên phải
    best = 0
    for i in range(m):
        cur_left, cur_right = 0, n
        for j in range(n):                       # height
            height[j] = height[j] + 1 if matrix[i][j] == "1" else 0
        for j in range(n):                       # left: trái -> phải
            if matrix[i][j] == "1":
                left[j] = max(left[j], cur_left)
            else:
                left[j] = 0
                cur_left = j + 1
        for j in range(n - 1, -1, -1):           # right: phải -> trái
            if matrix[i][j] == "1":
                right[j] = min(right[j], cur_right)
            else:
                right[j] = n
                cur_right = j
        for j in range(n):                       # diện tích
            best = max(best, (right[j] - left[j]) * height[j])
    return best

print(maximal_rectangle_dp_hlr(matrix1))   # 6
print(maximal_rectangle_dp_hlr(matrix2))   # 0
print(maximal_rectangle_dp_hlr(matrix3))   # 1

6
0
1


### Trace bảng cách 4 trên example 1

`n = 5`, khởi tạo `height = [0]*5`, `left = [0]*5`, `right = [5]*5`.

| hàng | ô | height | left | right | (right-left)*height | best |
|---:|---|---|---|---|---|---:|
| 0 | `1 0 1 0 0` | `[1,0,1,0,0]` | `[0,0,2,0,0]` | `[1,5,3,5,5]` | `[1,0,1,0,0]` | 1 |
| 1 | `1 0 1 1 1` | `[2,0,2,1,1]` | `[0,0,2,2,2]` | `[1,5,3,5,5]` | `[2,0,2,3,3]` | 3 |
| 2 | `1 1 1 1 1` | `[3,1,3,2,2]` | `[0,0,2,2,2]` | `[1,5,3,5,5]` | `[3,5,3,6,6]` | **6** |
| 3 | `1 0 0 1 0` | `[4,0,0,3,0]` | `[0,0,0,3,0]` | `[1,5,5,4,5]` | `[4,0,0,3,0]` | 6 |

Nhìn kỹ hàng 2, cột 3: `height = 2`, `left = 2`, `right = 5` → thanh cao 2 lan được từ cột 2 tới cột 4 → diện tích `3 * 2 = 6`.

Vì sao `left[3] = 2` dù hàng 2 toàn `1` (`cur_left = 0`)? Vì `left[3]` được **giữ từ hàng 1** (đoạn `1` ở hàng 1 bắt đầu từ cột 2), và `max(2, 0) = 2`: thanh cao 2 phải nằm gọn trong **cả** hàng 1 lẫn hàng 2.

Thú vị: ở hàng 2, `left` = `prev_smaller + 1` và `right` = `next_smaller` của cách 3 (so với output `largest_rectangle_two_pass` ở trên: `[-1,-1,1,1,1] + 1 = [0,0,2,2,2]` và `[1,5,3,5,5]`). Hai cách tính **cùng một thứ** bằng hai con đường khác nhau.

In [22]:
def trace_dp_hlr(matrix):
    m, n = len(matrix), len(matrix[0])
    height, left, right = [0] * n, [0] * n, [n] * n
    best = 0
    for i in range(m):
        cur_left, cur_right = 0, n
        for j in range(n):
            height[j] = height[j] + 1 if matrix[i][j] == "1" else 0
        for j in range(n):
            if matrix[i][j] == "1":
                left[j] = max(left[j], cur_left)
            else:
                left[j], cur_left = 0, j + 1
        for j in range(n - 1, -1, -1):
            if matrix[i][j] == "1":
                right[j] = min(right[j], cur_right)
            else:
                right[j], cur_right = n, j
        areas = [(right[j] - left[j]) * height[j] for j in range(n)]
        best = max(best, max(areas))
        print(f"hàng {i} {''.join(matrix[i])}: height={height} left={left} right={right} area={areas} best={best}")
    return best

print("Kết quả:", trace_dp_hlr(matrix1))

hàng 0 10100: height=[1, 0, 1, 0, 0] left=[0, 0, 2, 0, 0] right=[1, 5, 3, 5, 5] area=[1, 0, 1, 0, 0] best=1
hàng 1 10111: height=[2, 0, 2, 1, 1] left=[0, 0, 2, 2, 2] right=[1, 5, 3, 5, 5] area=[2, 0, 2, 3, 3] best=3
hàng 2 11111: height=[3, 1, 3, 2, 2] left=[0, 0, 2, 2, 2] right=[1, 5, 3, 5, 5] area=[3, 5, 3, 6, 6] best=6
hàng 3 10010: height=[4, 0, 0, 3, 0] left=[0, 0, 0, 3, 0] right=[1, 5, 5, 4, 5] area=[4, 0, 0, 3, 0] best=6
Kết quả: 6


### Độ phức tạp cách 4

- Mỗi hàng: 4 lượt duyệt `O(n)` → **Time `O(mn)`**.
- **Space `O(n)`**: ba mảng `height`, `left`, `right`.

So với cách 3: cùng Big-O, không dùng stack nên code "phẳng" (chỉ là vài vòng `for`), nhưng **khó tự nghĩ ra** và khó giải thích hơn. Trong phỏng vấn, cách 3 (quy về LeetCode 84) thường dễ trình bày và dễ thuyết phục hơn.

## 7. Code nộp LeetCode

Bản submit dùng **cách 3** (histogram + monotonic stack). Không `print`, đúng signature `maximalRectangle(self, matrix: List[List[str]]) -> int`. Hàm phụ `_largest_in_histogram` đặt trong class cho gọn.

In [23]:
class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        if not matrix or not matrix[0]:
            return 0
        n = len(matrix[0])
        heights = [0] * (n + 1)          # phần tử cuối luôn là 0: sentinel
        best = 0
        for row in matrix:
            for j in range(n):
                heights[j] = heights[j] + 1 if row[j] == "1" else 0
            best = max(best, self._largest_in_histogram(heights))
        return best

    def _largest_in_histogram(self, heights: List[int]) -> int:
        stack = []
        best = 0
        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:
                top = stack.pop()
                width = i - stack[-1] - 1 if stack else i
                best = max(best, heights[top] * width)
            stack.append(i)
        return best


sol = Solution()
tests = [
    (matrix1, 6),                                         # example 1
    ([["0"]], 0),                                         # example 2
    ([["1"]], 1),                                         # example 3
    ([["0", "0"], ["0", "0"]], 0),                        # toàn 0
    ([["1", "1"], ["1", "1"]], 4),                        # toàn 1
    ([["1", "1", "1", "1"]], 4),                          # một hàng
    ([["1"], ["1"], ["0"], ["1"]], 2),                    # một cột, bị cắt bởi 0
    ([["0", "1"], ["1", "0"]], 1),                        # đường chéo: chỉ được 1 ô
    ([["1", "0", "1"], ["1", "1", "1"], ["1", "1", "1"]], 6),
    ([["1"] * 200 for _ in range(200)], 40000),           # cỡ lớn nhất theo constraints
]
for mat, expected in tests:
    got = sol.maximalRectangle(mat)
    assert got == expected, (mat, got, expected)
    label = f"{len(mat)}x{len(mat[0])}"
    print(f"{label:>8} -> {got:>6} PASS")

     4x5 ->      6 PASS
     1x1 ->      0 PASS
     1x1 ->      1 PASS
     2x2 ->      0 PASS
     2x2 ->      4 PASS
     1x4 ->      4 PASS
     4x1 ->      2 PASS
     2x2 ->      1 PASS
     3x3 ->      6 PASS
 200x200 ->  40000 PASS


### Bản nộp thay thế: cách 4 (DP `height/left/right`)

Nếu muốn nộp cách 4, đổi tên class `SolutionDP` thành `Solution` khi dán lên LeetCode.

In [24]:
class SolutionDP:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        m, n = len(matrix), len(matrix[0])
        height, left, right = [0] * n, [0] * n, [n] * n
        best = 0
        for i in range(m):
            row = matrix[i]
            cur_left, cur_right = 0, n
            for j in range(n):
                if row[j] == "1":
                    height[j] += 1
                    left[j] = max(left[j], cur_left)
                else:
                    height[j], left[j], cur_left = 0, 0, j + 1
            for j in range(n - 1, -1, -1):
                if row[j] == "1":
                    right[j] = min(right[j], cur_right)
                else:
                    right[j], cur_right = n, j
            for j in range(n):
                best = max(best, (right[j] - left[j]) * height[j])
        return best


for mat, expected in tests:
    assert SolutionDP().maximalRectangle(mat) == expected
print("SolutionDP: PASS toàn bộ", len(tests), "test")

SolutionDP: PASS toàn bộ 10 test


### Kiểm thử đối chiếu chéo tất cả các cách

Brute force chậm nhưng **dễ tin là đúng** vì làm đúng theo định nghĩa. Ta sinh ngẫu nhiên nhiều ma trận nhỏ (cố định `random.seed` để lần chạy nào cũng ra cùng bộ test), thay đổi cả **mật độ** số `1` để có ma trận thưa lẫn dày, rồi so mọi cách với nhau. Sau đó đo thời gian trên ma trận `200 x 200`.

In [25]:
def random_matrix(m, n, p):
    return [["1" if random.random() < p else "0" for _ in range(n)] for _ in range(m)]


random.seed(85)
for _ in range(800):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = random_matrix(m, n, random.choice([0.3, 0.6, 0.85, 1.0]))
    expected = maximal_rectangle_bruteforce(mat)
    assert maximal_rectangle_prefix2d(mat) == expected, mat
    assert maximal_rectangle_width_dp(mat) == expected, mat
    assert maximal_rectangle_histogram(mat) == expected, mat
    assert maximal_rectangle_dp_hlr(mat) == expected, mat
    assert Solution().maximalRectangle(mat) == expected, mat
    assert SolutionDP().maximalRectangle(mat) == expected, mat
print("800 ma trận ngẫu nhiên: cả 4 cách (và 2 class) cho cùng kết quả!")

for p in [0.7, 1.0]:
    big = random_matrix(200, 200, p)
    results = []
    for f in [maximal_rectangle_width_dp, maximal_rectangle_histogram, maximal_rectangle_dp_hlr]:
        t0 = time.perf_counter()
        results.append(f(big))
        print(f"  p={p}: {f.__name__:<28} = {results[-1]:>6}  ({time.perf_counter() - t0:.2f}s)")
    assert len(set(results)) == 1

800 ma trận ngẫu nhiên: cả 4 cách (và 2 class) cho cùng kết quả!


  p=0.7: maximal_rectangle_width_dp   =     35  (0.02s)
  p=0.7: maximal_rectangle_histogram  =     35  (0.01s)


  p=0.7: maximal_rectangle_dp_hlr     =     35  (0.01s)


  p=1.0: maximal_rectangle_width_dp   =  40000  (0.42s)
  p=1.0: maximal_rectangle_histogram  =  40000  (0.01s)
  p=1.0: maximal_rectangle_dp_hlr     =  40000  (0.01s)


Bạn sẽ thấy (thời gian cụ thể tùy máy): với ma trận **toàn `1`** (`p = 1.0`), cách 2 chậm hẳn vì mỗi ô phải lan lên tận hàng 0 (đúng worst case `O(m²n)`), còn cách 3 và 4 vẫn nhanh như nhau (`O(mn)`). Còn cách 1 (brute force) thì không ai dám chạy ở cỡ này.

## 8. Vì sao đúng? (invariant bằng lời)

**Cách 3 — histogram:**

1. *Không bỏ sót:* lấy hình chữ nhật tối ưu bất kỳ, gọi hàng dưới cùng của nó là `i`, cao `h`, chiếm cột `c1..c2`. Khi xét đáy hàng `i`, mọi cột `j ∈ [c1, c2]` có `heights[j] >= h` (vì có ít nhất `h` ô `1` liên tiếp kéo lên). Gọi `j*` là cột thấp nhất trong đoạn đó. Hình chữ nhật mà thuật toán tính cho `j*` cao `heights[j*] >= h` và rộng ít nhất `c2 - c1 + 1` (vì nó lan tới khi gặp cột **thấp hơn** `heights[j*]`, mà trong `[c1, c2]` không có cột nào như vậy). Nên diện tích tìm được `>=` diện tích tối ưu.
2. *Không trả nhầm:* mọi diện tích thuật toán tính đều ứng với một hình chữ nhật thật trong histogram, tức một khối toàn `1` thật trong ma trận.

**Monotonic stack (invariant):** tại mọi thời điểm, chiều cao các index trong stack **không giảm** từ đáy lên đỉnh, và với mỗi phần tử trong stack, phần tử ngay dưới nó là **cột gần nhất bên trái thấp hơn hẳn** nó (các cột ở giữa đã bị pop vì cao hơn hoặc bằng nó). Khi `top` bị `i` pop, `i` là cột đầu tiên bên phải thấp hơn nó. Vậy ta có đủ hai biên → width đúng.

**Cách 4 — DP:** invariant sau khi xử lý hàng `i`: nếu `height[j] > 0` thì `left[j]..right[j]-1` là **đoạn cột rộng nhất chứa `j`** sao cho mọi ô ở các hàng `i-height[j]+1 .. i` trong đoạn đó đều là `1`. Lý do: đoạn này là **giao** của các đoạn `1` chứa `j` ở từng hàng đó; `max` của các biên trái và `min` của các biên phải chính là phép lấy giao. Lập luận "cột thấp nhất `j*`" như cách 3 cho thấy hình tối ưu cũng được xét tới.

## 9. So sánh các cách

`m` = số hàng, `n` = số cột.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---:|---:|---|
| 1a. Brute force | Thử mọi hình, kiểm từng ô | `O(m³n³)` | `O(1)` | Chuẩn vàng để test, quá chậm |
| 1b. Brute + prefix 2D | Kiểm tra hình trong `O(1)` | `O(m²n²)` | `O(mn)` | Vẫn TLE ở `200 x 200` |
| 2. DP `width` | Cố định góc dưới-phải, lan lên lấy `min` | `O(m²n)` | `O(mn)` | Qua được, dễ hiểu, chậm khi nhiều `1` |
| 3. Histogram + stack | Mỗi hàng là đáy histogram, giải LC 84 | `O(mn)` | `O(n)` | **Cách chính**, dễ trình bày |
| 4. DP height/left/right | Cập nhật biên trái/phải theo hàng | `O(mn)` | `O(n)` | Không stack, khó tự nghĩ ra |

Lộ trình suy nghĩ: **thử hết** → **kiểm tra nhanh hơn (prefix sum)** → **tái sử dụng độ rộng (DP width)** → **đổi góc nhìn sang chiều cao (histogram)** → **công cụ "phần tử nhỏ hơn gần nhất" (monotonic stack)** hoặc **DP biên trái/phải**.

## 10. Lỗi người mới hay gặp

### Lỗi 1: Coi chuỗi `'0'` là `False`

Sai: `heights[j] = heights[j] + 1 if row[j] else 0`. Chuỗi `"0"` là truthy → mọi ô đều được coi là `1` → đáp án luôn là `m * n`. Sửa: `row[j] == "1"`.

### Lỗi 2: Quên reset `heights[j]` về 0 khi gặp `'0'`

Sai: `heights[j] += (row[j] == "1")`. Cột bị cắt bởi một ô `0` mà vẫn cộng dồn → coi như khối `1` "nhảy qua" ô `0`. Sửa: gặp `'0'` thì `heights[j] = 0`.

### Lỗi 3: Quên sentinel ở cuối histogram

Sai: không nối `0` vào cuối. Những cột còn kẹt trong stack (ví dụ histogram tăng dần `[1, 2, 3]`) không bao giờ bị pop → không được tính diện tích. Sửa: duyệt `heights + [0]` (hoặc xử lý nốt stack sau vòng lặp).

### Lỗi 4: Sai width khi pop

Sai: `width = i - stack[-1]` (thiếu `- 1`), hoặc khi stack rỗng lại dùng `i - 1`. Biên trái `stack[-1]` và biên phải `i` đều **thấp hơn**, không thuộc hình chữ nhật; số cột ở giữa là `i - stack[-1] - 1`. Stack rỗng nghĩa là không có biên trái → hình kéo dài từ cột 0 tới `i - 1`, rộng đúng `i`.

### Lỗi 5 (cách 4): Không reset `left[j] = 0` ở ô `'0'`

Sai: ở ô `0` chỉ cập nhật `cur_left` mà giữ nguyên `left[j]`. Hàng sau, thanh mới ở cột `j` bị chặn oan bởi biên trái **cũ** → có thể **thiếu** đáp án.

Cell dưới chạy thử các bản sai. Riêng lỗi 5, ta để máy **tự tìm phản ví dụ** bằng cách thử ngẫu nhiên ma trận nhỏ — đây là kỹ thuật **stress test** rất đáng học.

In [26]:
def wrong_truthy(matrix):                       # Lỗi 1
    n = len(matrix[0]); heights = [0] * n; best = 0
    for row in matrix:
        for j in range(n):
            heights[j] = heights[j] + 1 if row[j] else 0      # SAI: '0' là truthy
        best = max(best, largest_rectangle_histogram(heights))
    return best

def wrong_no_reset(matrix):                     # Lỗi 2
    n = len(matrix[0]); heights = [0] * n; best = 0
    for row in matrix:
        for j in range(n):
            heights[j] += (row[j] == "1")                     # SAI: không về 0
        best = max(best, largest_rectangle_histogram(heights))
    return best

def wrong_histogram_no_sentinel(heights):       # Lỗi 3
    stack, best = [], 0
    for i, h in enumerate(heights):                           # SAI: không có sentinel
        while stack and heights[stack[-1]] > h:
            top = stack.pop()
            width = i - stack[-1] - 1 if stack else i
            best = max(best, heights[top] * width)
        stack.append(i)
    return best

def wrong_histogram_width(heights):             # Lỗi 4
    ext, stack, best = heights + [0], [], 0
    for i, h in enumerate(ext):
        while stack and ext[stack[-1]] > h:
            top = stack.pop()
            width = i - stack[-1] if stack else i             # SAI: thiếu -1
            best = max(best, ext[top] * width)
        stack.append(i)
    return best

print("đúng                       :", maximal_rectangle_histogram(matrix1))
print("lỗi 1 (truthy '0')         :", wrong_truthy(matrix1))
cut = [["1", "1"], ["0", "0"], ["1", "1"]]       # hàng giữa toàn 0 "cắt đứt" hai hàng 1
print("lỗi 2 (không reset heights) trên", cut, ":", wrong_no_reset(cut),
      "| đúng:", maximal_rectangle_histogram(cut))
print("lỗi 3 (không sentinel) [1,2,3]:", wrong_histogram_no_sentinel([1, 2, 3]),
      "| đúng:", largest_rectangle_histogram([1, 2, 3]))
print("lỗi 4 (width thiếu -1) [3,1,3,2,2]:", wrong_histogram_width([3, 1, 3, 2, 2]),
      "| đúng:", largest_rectangle_histogram([3, 1, 3, 2, 2]))

đúng                       : 6
lỗi 1 (truthy '0')         : 20
lỗi 2 (không reset heights) trên [['1', '1'], ['0', '0'], ['1', '1']] : 4 | đúng: 2
lỗi 3 (không sentinel) [1,2,3]: 0 | đúng: 4
lỗi 4 (width thiếu -1) [3,1,3,2,2]: 8 | đúng: 6


In [27]:
def wrong_dp_no_left_reset(matrix):             # Lỗi 5
    m, n = len(matrix), len(matrix[0])
    height, left, right = [0] * n, [0] * n, [n] * n
    best = 0
    for i in range(m):
        cur_left, cur_right = 0, n
        for j in range(n):
            if matrix[i][j] == "1":
                height[j] += 1
                left[j] = max(left[j], cur_left)
            else:
                height[j], cur_left = 0, j + 1                # SAI: quên left[j] = 0
        for j in range(n - 1, -1, -1):
            if matrix[i][j] == "1":
                right[j] = min(right[j], cur_right)
            else:
                right[j], cur_right = n, j
        for j in range(n):
            best = max(best, (right[j] - left[j]) * height[j])
    return best


# Stress test: thử ngẫu nhiên cho tới khi bản sai lệch với bản đúng
random.seed(5)
for attempt in range(1, 100001):
    mat = random_matrix(random.randint(1, 4), random.randint(1, 4), 0.6)
    if wrong_dp_no_left_reset(mat) != maximal_rectangle_bruteforce(mat):
        print(f"Tìm thấy phản ví dụ sau {attempt} lần thử:")
        show_matrix(mat)
        print("bản sai trả về:", wrong_dp_no_left_reset(mat), "| đúng:", maximal_rectangle_bruteforce(mat))
        break

Tìm thấy phản ví dụ sau 40 lần thử:
    0 1 2 3
0 | . . 1 .
1 | . . 1 .
2 | 1 1 . 1
3 | . 1 1 1
bản sai trả về: 2 | đúng: 3


## 11. Mẹo nhận dạng pattern

- Ma trận `0/1` + hỏi **hình chữ nhật / hình vuông toàn `1` lớn nhất** hoặc **đếm số hình chữ nhật toàn `1`** → nghĩ ngay tới **"mỗi hàng là đáy, tính `heights` theo cột"**. Đây là cây cầu 2D → 1D.
- Câu hỏi kiểu "với mỗi phần tử, tìm phần tử **nhỏ hơn / lớn hơn gần nhất** bên trái/phải" → **monotonic stack**, `O(n)`.
- Hình chữ nhật trong histogram: "**giả sử cột này là cột thấp nhất**, nó lan được bao xa?" — kỹ thuật **cố định phần tử nhỏ nhất** (fix the minimum), dùng được cho rất nhiều bài kiểu "max của `min(...) * độ dài`".
- Thấy công thức có `min` khi **mở rộng một đoạn/khối** và cách lan từng bước chậm → hỏi "có lan được **một lần cho cả hàng** không?".
- Prefix sum 2D: hễ cần **hỏi tổng của nhiều hình chữ nhật con** trong ma trận cố định → dựng bảng `P` một lần, mỗi câu hỏi `O(1)`.

## 12. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 84 — Largest Rectangle in Histogram | Chính là bài con của cách 3 |
| LeetCode 221 — Maximal Square | Hình **vuông** toàn `1`: có DP `O(mn)` đơn giản `dp = min(trên, trái, chéo) + 1` (bài tập 5) |
| LeetCode 1504 — Count Submatrices With All Ones | **Đếm** hình chữ nhật toàn `1`: dùng ngay DP `width` của cách 2 (bài tập 6) |
| LeetCode 1727 — Largest Submatrix With Rearrangements | Được hoán đổi các cột: tính `heights` rồi **sắp xếp** mỗi hàng |
| LeetCode 304 — Range Sum Query 2D - Immutable | Prefix sum 2D thuần túy |
| LeetCode 42 — Trapping Rain Water | Cũng là bài histogram, có lời giải monotonic stack |
| LeetCode 907 — Sum of Subarray Minimums | "Cố định phần tử nhỏ nhất" + monotonic stack |

## 13. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 14, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay `heights` (không chạy code)

Cho ma trận:

```text
1 1 0 1
1 1 1 1
0 1 1 1
1 1 1 0
```

(a) Viết `heights` sau mỗi hàng đáy 0, 1, 2, 3.
(b) Với mỗi hàng đáy, hình chữ nhật lớn nhất trong histogram có diện tích bao nhiêu?
(c) Đáp án của bài là bao nhiêu? Chỉ ra một hình chữ nhật đạt được (tọa độ `r1, c1, r2, c2`).

### Bài 2 — Đoán thứ tự pop của monotonic stack

Chạy tay `largest_rectangle_histogram([2, 1, 5, 6, 2, 3])` (có sentinel `0` ở cuối). Liệt kê **theo thứ tự** các lần pop: index nào bị pop, ở bước `i` nào, width và diện tích bao nhiêu. Kết quả cuối là gì? Hình chữ nhật lớn nhất nằm ở các cột nào?

### Bài 3 — Điền TODO: hình chữ nhật lớn nhất trong histogram

Hoàn thiện hàm theo chú thích. Nhớ: sentinel, điều kiện pop, công thức width.

In [28]:
def exercise_largest_rectangle(heights: List[int]) -> int:
    stack = []
    best = 0
    extended = heights + [0]
    for i, h in enumerate(extended):
        # TODO 1: trong khi stack khác rỗng và cột ở đỉnh CAO HƠN h:
        #     TODO 2: pop index top
        #     TODO 3: tính width (stack rỗng -> i, ngược lại -> i - stack[-1] - 1)
        #     TODO 4: cập nhật best bằng extended[top] * width
        # TODO 5: push i
        pass
    return best

# Tự làm xong rồi mới mở test:
# assert exercise_largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
# assert exercise_largest_rectangle([2, 4]) == 4
# assert exercise_largest_rectangle([3, 1, 3, 2, 2]) == 6
# assert exercise_largest_rectangle([0]) == 0

### Bài 4 — Tự viết lại cách 3 từ đầu

Không nhìn lên trên, tự viết `exercise_maximal_rectangle(matrix)` (một hàm duy nhất, có thể viết hàm histogram lồng bên trong). Hãy chú ý bẫy chuỗi `'0'`.

In [29]:
def exercise_maximal_rectangle(matrix: List[List[str]]) -> int:
    # TODO: heights theo từng hàng đáy + monotonic stack cho mỗi histogram
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_maximal_rectangle(matrix1) == 6
# assert exercise_maximal_rectangle([["0"]]) == 0
# assert exercise_maximal_rectangle([["1"]]) == 1
# assert exercise_maximal_rectangle([["1", "1"], ["1", "0"]]) == 2

### Bài 5 — Biến thể: LeetCode 221 — Maximal Square

Cùng đầu vào, nhưng tìm **hình vuông** toàn `'1'` lớn nhất, trả về diện tích.

- `[["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]` → `4`
- `[["0","1"],["1","0"]]` → `1`
- `[["0"]]` → `0`

Gợi ý hướng: đặt `side[i][j]` = cạnh hình vuông toàn `1` lớn nhất có **góc dưới-phải** tại `(i, j)`. Tìm công thức theo 3 ô hàng xóm.

In [30]:
def exercise_maximal_square(matrix: List[List[str]]) -> int:
    # TODO: DP side[i][j] cho góc dưới-phải (i, j)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_maximal_square(matrix1) == 4
# assert exercise_maximal_square([["0", "1"], ["1", "0"]]) == 1
# assert exercise_maximal_square([["0"]]) == 0

### Bài 6 — Mini challenge: LeetCode 1504 — Count Submatrices With All Ones

Cho ma trận `mat` gồm **số nguyên** `0`/`1` (để ý: bài này là số, không phải chuỗi). Đếm **số hình chữ nhật con** chỉ chứa `1`.

- `[[1,0,1],[1,1,0],[1,1,0]]` → `13`
- `[[0,1,1,0],[0,1,1,1],[1,1,1,0]]` → `24`

Gợi ý hướng: dùng lại DP `width` của cách 2. Với mỗi góc dưới-phải `(i, j)`, lan lên trên giữ `min_w`; số hình chữ nhật có góc dưới-phải `(i, j)` và hàng trên cùng là `k` bằng đúng `min_w`.

In [31]:
def exercise_count_submatrices(mat: List[List[int]]) -> int:
    # TODO: width[i][j] + lan lên trên, cộng min_w mỗi bước
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_count_submatrices([[1, 0, 1], [1, 1, 0], [1, 1, 0]]) == 13
# assert exercise_count_submatrices([[0, 1, 1, 0], [0, 1, 1, 1], [1, 1, 1, 0]]) == 24

## 14. Gợi ý theo mức

**Bài 1**
- Mức 1: làm như bảng ở mục 5: `+1` nếu `1`, về `0` nếu `0`.
- Mức 2: hàng đáy 2 có ô `(2, 0) = 0` nên `heights[0]` về 0.
- Mức 3: `heights` lần lượt là `[1,1,0,1]`, `[2,2,1,2]`, ... — tự tính tiếp hai hàng còn lại, rồi dùng "cột thấp nhất lan được bao xa" cho từng histogram.

**Bài 2**
- Mức 1: chỉ pop khi gặp cột **thấp hơn** đỉnh stack. Các bước `i = 2, 3` đều cao hơn → chỉ push.
- Mức 2: ở `i = 4` (cao 2) sẽ pop hai lần; ở `i = 6` (sentinel) pop ba lần.
- Mức 3: lần pop ở `i = 4` thứ hai là index 2 (cao 5), biên trái là index 1 → width `4 - 1 - 1 = 2`.

**Bài 3**
- Mức 1: chép khung `while stack and ...:` của mục 5.
- Mức 2: điều kiện là `extended[stack[-1]] > h`.
- Mức 3: `top = stack.pop(); width = i - stack[-1] - 1 if stack else i; best = max(best, extended[top] * width)`.

**Bài 4**
- Mức 1: hai phần: cập nhật `heights` cho hàng, rồi gọi histogram.
- Mức 2: dùng `heights = [0] * (n + 1)` để phần tử cuối luôn là sentinel 0.
- Mức 3: `heights[j] = heights[j] + 1 if row[j] == "1" else 0`.

**Bài 5**
- Mức 1: hình vuông cạnh `k` có góc dưới-phải `(i, j)` cần hình vuông cạnh `k - 1` ở **trên**, **trái**, **chéo trên-trái**.
- Mức 2: `side[i][j] = min(side[i-1][j], side[i][j-1], side[i-1][j-1]) + 1` nếu ô là `'1'`, ngược lại `0`.
- Mức 3: dùng bảng `(m+1) x (n+1)` để khỏi xử lý biên; trả về `max_side ** 2` (diện tích, không phải cạnh!).

**Bài 6**
- Mức 1: số hình chữ nhật có góc dưới-phải `(i, j)` và **chỉ một hàng** là `width[i][j]` (chọn cột trái từ `j - width + 1` tới `j`).
- Mức 2: thêm hàng phía trên `k`: số cột trái hợp lệ là `min_w = min(width[k..i][j])`.
- Mức 3: `for k in range(i, -1, -1): min_w = min(min_w, width[k][j]); if min_w == 0: break; total += min_w`.

## 15. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1**

| đáy hàng | heights | lớn nhất trong histogram |
|---:|---|---:|
| 0 | `[1, 1, 0, 1]` | 2 |
| 1 | `[2, 2, 1, 2]` | 4 |
| 2 | `[0, 3, 2, 3]` | 6 |
| 3 | `[1, 4, 3, 0]` | 6 |

Đáp án **6**. Ví dụ hình `(1, 1)-(3, 2)`: hàng 1..3, cột 1..2 (cao 3, rộng 2). Hình `(1, 1)-(2, 3)` (cao 2, rộng 3) cũng đạt 6.

**Bài 2** — `[2, 1, 5, 6, 2, 3]` + sentinel `0`:

| i | pop (h) | biên trái | width | diện tích |
|---:|---|---|---:|---:|
| 1 | 0 (2) | rỗng | 1 | 2 |
| 4 | 3 (6) | 2 | 1 | 6 |
| 4 | 2 (5) | 1 | 2 | **10** |
| 6 | 5 (3) | 4 | 1 | 3 |
| 6 | 4 (2) | 1 | 4 | 8 |
| 6 | 1 (1) | rỗng | 6 | 6 |

Kết quả **10**: hình cao 5 trên cột 2..3.

Cell dưới kiểm chứng bài 1 và bài 2 bằng máy (dùng các hàm đã có ở trên), sau đó là đáp án code bài 3–6.

In [32]:
# Kiểm chứng Bài 1
ex1 = [list("1101"), list("1111"), list("0111"), list("1110")]
trace_maximal_rectangle_histogram(ex1)
show_matrix(ex1, rect=(1, 1, 3, 2))
print()
# Kiểm chứng Bài 2
print("Kết quả:", trace_histogram_stack([2, 1, 5, 6, 2, 3]))

đáy hàng 0: 1101 -> heights=[1, 1, 0, 1], lớn nhất trong histogram=2, best=2
đáy hàng 1: 1111 -> heights=[2, 2, 1, 2], lớn nhất trong histogram=4, best=4
đáy hàng 2: 0111 -> heights=[0, 3, 2, 3], lớn nhất trong histogram=6, best=6
đáy hàng 3: 1110 -> heights=[1, 4, 3, 0], lớn nhất trong histogram=6, best=6
    0 1 2 3
0 | 1 1 . 1
1 | 1 # # 1
2 | . # # 1
3 | 1 # # .

heights + sentinel = [2, 1, 5, 6, 2, 3, 0]
  i=0 h=2: push -> stack = [0]
  i=1 h=1: pop 0 (h=2), biên trái=-1, width=1, diện tích=2, best=2
  i=1 h=1: push -> stack = [1]
  i=2 h=5: push -> stack = [1, 2]
  i=3 h=6: push -> stack = [1, 2, 3]
  i=4 h=2: pop 3 (h=6), biên trái= 2, width=1, diện tích=6, best=6
  i=4 h=2: pop 2 (h=5), biên trái= 1, width=2, diện tích=10, best=10
  i=4 h=2: push -> stack = [1, 4]
  i=5 h=3: push -> stack = [1, 4, 5]
  i=6 h=0: pop 5 (h=3), biên trái= 4, width=1, diện tích=3, best=10
  i=6 h=0: pop 4 (h=2), biên trái= 1, width=4, diện tích=8, best=10
  i=6 h=0: pop 1 (h=1), biên trái=-1, width=6

In [33]:
def answer_largest_rectangle(heights: List[int]) -> int:          # Bài 3
    stack = []
    best = 0
    extended = heights + [0]
    for i, h in enumerate(extended):
        while stack and extended[stack[-1]] > h:
            top = stack.pop()
            width = i - stack[-1] - 1 if stack else i
            best = max(best, extended[top] * width)
        stack.append(i)
    return best


def answer_maximal_rectangle(matrix: List[List[str]]) -> int:      # Bài 4
    n = len(matrix[0])
    heights = [0] * (n + 1)                # heights[n] = 0 là sentinel cố định

    def largest(hs):
        stack, best = [], 0
        for i, h in enumerate(hs):
            while stack and hs[stack[-1]] > h:
                top = stack.pop()
                width = i - stack[-1] - 1 if stack else i
                best = max(best, hs[top] * width)
            stack.append(i)
        return best

    best = 0
    for row in matrix:
        for j in range(n):
            heights[j] = heights[j] + 1 if row[j] == "1" else 0
        best = max(best, largest(heights))
    return best


assert answer_largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
assert answer_largest_rectangle([2, 4]) == 4
assert answer_largest_rectangle([3, 1, 3, 2, 2]) == 6
assert answer_largest_rectangle([0]) == 0
assert answer_maximal_rectangle(matrix1) == 6
assert answer_maximal_rectangle([["0"]]) == 0
assert answer_maximal_rectangle([["1"]]) == 1
assert answer_maximal_rectangle([["1", "1"], ["1", "0"]]) == 2
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


In [34]:
def answer_maximal_square(matrix: List[List[str]]) -> int:         # Bài 5
    m, n = len(matrix), len(matrix[0])
    side = [[0] * (n + 1) for _ in range(m + 1)]   # side[i+1][j+1] ứng với ô (i, j)
    max_side = 0
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == "1":
                side[i + 1][j + 1] = min(side[i][j + 1], side[i + 1][j], side[i][j]) + 1
                max_side = max(max_side, side[i + 1][j + 1])
    return max_side * max_side


def answer_count_submatrices(mat: List[List[int]]) -> int:         # Bài 6
    m, n = len(mat), len(mat[0])
    width = [[0] * n for _ in range(m)]
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 1:
                width[i][j] = (width[i][j - 1] if j > 0 else 0) + 1
            min_w = width[i][j]
            for k in range(i, -1, -1):
                min_w = min(min_w, width[k][j])
                if min_w == 0:
                    break
                total += min_w
    return total


assert answer_maximal_square(matrix1) == 4
assert answer_maximal_square([["0", "1"], ["1", "0"]]) == 1
assert answer_maximal_square([["0"]]) == 0
assert answer_count_submatrices([[1, 0, 1], [1, 1, 0], [1, 1, 0]]) == 13
assert answer_count_submatrices([[0, 1, 1, 0], [0, 1, 1, 1], [1, 1, 1, 0]]) == 24

# Đối chiếu ngẫu nhiên các đáp án với brute force
def count_submatrices_bruteforce(mat):
    m, n = len(mat), len(mat[0])
    return sum(all(mat[i][j] == 1 for i in range(r1, r2 + 1) for j in range(c1, c2 + 1))
               for r1 in range(m) for c1 in range(n) for r2 in range(r1, m) for c2 in range(c1, n))

def maximal_square_bruteforce(matrix):
    m, n = len(matrix), len(matrix[0])
    best = 0
    for r in range(m):
        for c in range(n):
            k = 1
            while r + k <= m and c + k <= n and all(matrix[i][j] == "1"
                                                    for i in range(r, r + k) for j in range(c, c + k)):
                best = max(best, k * k)
                k += 1
    return best

random.seed(2024)
for _ in range(400):
    mat = random_matrix(random.randint(1, 5), random.randint(1, 5), random.choice([0.5, 0.8, 1.0]))
    assert answer_maximal_rectangle(mat) == maximal_rectangle_bruteforce(mat)
    assert answer_maximal_square(mat) == maximal_square_bruteforce(mat)
    int_mat = [[int(ch) for ch in row] for row in mat]
    assert answer_count_submatrices(int_mat) == count_submatrices_bruteforce(int_mat)
print("Bài 5, Bài 6 và đối chiếu ngẫu nhiên: PASS toàn bộ!")

Bài 5, Bài 6 và đối chiếu ngẫu nhiên: PASS toàn bộ!


## 16. Template ngắn nên nhớ

```python
class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        n = len(matrix[0])
        heights = [0] * (n + 1)                 # heights[n] = 0: sentinel
        best = 0
        for row in matrix:
            for j in range(n):                  # 2D -> 1D: mỗi hàng là đáy histogram
                heights[j] = heights[j] + 1 if row[j] == "1" else 0
            stack = []                          # monotonic stack (chiều cao tăng dần)
            for i, h in enumerate(heights):
                while stack and heights[stack[-1]] > h:
                    top = stack.pop()           # top gặp biên phải i, biên trái là stack[-1]
                    width = i - stack[-1] - 1 if stack else i
                    best = max(best, heights[top] * width)
                stack.append(i)
        return best
```

Câu thần chú: **"Mỗi hàng là một cái đáy; đếm chiều cao cột; cột nào bị pop thì biết biên phải là người pop nó, biên trái là người đứng dưới nó."**

## 17. Checklist trước khi submit

- So sánh ô bằng `== "1"` (chuỗi), **không** dùng `if row[j]:`?
- Gặp `'0'` thì `heights[j] = 0` (reset), không cộng dồn?
- Có sentinel `0` ở cuối histogram (hoặc xử lý nốt stack sau vòng lặp)?
- Điều kiện pop là đỉnh **cao hơn** `h`; width là `i - stack[-1] - 1`, stack rỗng thì `i`?
- Trả về **diện tích** (số ô), không phải chiều cao hay tọa độ?
- Đã thử 3 example của đề, ma trận toàn `0`, toàn `1`, một hàng, một cột, và một ma trận ngẫu nhiên so với brute force?